In [1]:
# assignment_4.ipynb Samuel Antonio Canedo Perez - NLP class spring 2026
# Use the provided server environment to train an RNN-based language model on the
# Shakespeare corpus.

In [2]:
# We start defining the pipeline for the RNN-base language model
# Aquisition of the data and the text processing

# We start by downloading the corpus and normalizing it

# We import the needed libraries
import re
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# We download the Shakespeare corpus
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
raw_text = requests.get(url).text

# We modify it to lower case
text = raw_text.lower()

# Quick verification of the data characteristics
print(f'Corpus lenght (characters): {len(text)}')
print('\n\nSample of the firts 150 characters:')
print('-' * 30)
print(text[:150])
print('-' * 30)


Corpus lenght (characters): 1115394


Sample of the firts 150 characters:
------------------------------
first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

a
------------------------------


In [3]:
# Tokenization & Vocabulary Building

# For this model we're builidn a world-level tokenizer instead of a character
# level tokenizer, thus with purpose we try to have a stronger grasp of English

# We start tokenizing the text usign Regular Expressions
# Extracting the sequens of alphanumeric characters (\w+)
# and separating individual punctuations marks ([^\w\s]) as distinct tokens
tokens = re.findall(r'\w+|[^\w\s]', text)

# We check the tokens
print(f'Total tokes extracted: {len(tokens)}')
print(f'First 20 tokens: {tokens[:20]}\n')

# We build the vocabulary, removing duplicates with set()
vocab = sorted(set(tokens))
vocab_size = len(vocab)

print(f'Unique Vocabulary Size: {vocab_size}')
print(f'Sample Vocabulary items (first 30): {vocab[:30]}\n')

# We create the encoding and decoding lookups
# stoi: string-to-index maps a word to a number for training
# itos: index-to-string maps a number back to a word for generating text
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}

# Encode the entire corpus into integers
# We wrap in the pytorch tensor for the dataset creation
encoded_data = torch.tensor([stoi[tok] for tok in tokens], dtype=torch.long)

print(f'Encoded data tensor shape: {encoded_data.shape}')
print(f'First 20 encoded token IDs: {encoded_data[:20].tolist()}')

Total tokes extracted: 262927
First 20 tokens: ['first', 'citizen', ':', 'before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'all', ':', 'speak', ',', 'speak', '.', 'first']

Unique Vocabulary Size: 11466
Sample Vocabulary items (first 30): ['!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'a', 'abandon', 'abase', 'abate', 'abated', 'abbey', 'abbot', 'abed', 'abel', 'abet', 'abhor', 'abhorr', 'abhorred', 'abhorring', 'abhors', 'abhorson', 'abide', 'abides', 'abilities']

Encoded data tensor shape: torch.Size([262927])
First 20 encoded token IDs: [3820, 1759, 8, 841, 11061, 7619, 413, 4177, 4, 4716, 6150, 9240, 6, 291, 8, 9240, 4, 9240, 6, 3820]


In [4]:
# Dataset Creation & Mini-batch DataLoader

# Creation of a custom PyThorch Dataset to slice the pairs, and feed the DataLoader

class ShakespeareWordDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        # Last sequence must stop early so its target window doesn't overflow
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # X is a chunk of history tokens
        x = self.data[idx : idx + self.seq_len]
        # Y is the exact same chunk shifted forward by one step
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y

# We define the sequence length and instantiate dataset
# window of 30 words for the context
seq_len = 30
dataset = ShakespeareWordDataset(encoded_data, seq_len)

# Construct the DataLoader
# We use a batch size of 64 and drop_last= True to keep batch matric shapes uniform
batch_size = 64
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Check on the shapes
sample_x, sample_y = next(iter(dataloader))
print(f'Total dataset samples: {len(dataset)}')
print(f"Mini-batch Input X shape:  {sample_x.shape}  -> (Batch Size, Sequence Length)")
print(f"Mini-batch Target Y shape: {sample_y.shape}  -> (Batch Size, Sequence Length)")
print("\nFirst sample pair matched visualization (Token IDs):")
print(f"X[0]: {sample_x[0].tolist()[:10]}...")
print(f"Y[0]: {sample_y[0].tolist()[:10]}... (Shifted right by 1)")

Total dataset samples: 262897
Mini-batch Input X shape:  torch.Size([64, 30])  -> (Batch Size, Sequence Length)
Mini-batch Target Y shape: torch.Size([64, 30])  -> (Batch Size, Sequence Length)

First sample pair matched visualization (Token IDs):
X[0]: [4, 6542, 6218, 3095, 9990, 11044, 5, 3918, 9910, 9]...
Y[0]: [6542, 6218, 3095, 9990, 11044, 5, 3918, 9910, 9, 5043]... (Shifted right by 1)


In [5]:
# Recurrent Neural Network (LSTM) Architecture Design

# We buld the memory for the RNN based in the LSTM achitecture

class WorldLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()

        # embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Recurrent layer
        self.rnn = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True # Indicates that the data come in batches
        )

        # Lineal Exit Layer (Clasificator)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x has an initial shape of; (Batch Size, Sequence Length) - [64, 30]

        # Convert the IDs of teh tokens in vectors
        x = self.embedding(x) # (B, T, Embed_Dim) - [64, 30, 128]

        # Process the sequence through LSTM
        out, hidden = self.rnn(x, hidden) # [64, 30, 256] hidden contains the internal states

        # Project the hidden space back to the vocab size
        logits = self.fc(out)

        return logits, hidden

# add teh hyperparameters to the model and use cuda as recommended in class
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

embed_dim = 128         # Each word is represented by a vector of 128 numbers
hidden_dim = 256        # Size of the intern memory of the LSTM
num_layers = 2          # We apply 2 LSTM on top of each other to capture complex patterns

# We construct the model and we send it to the GPU if available
model = WorldLSTM(vocab_size, embed_dim, hidden_dim, num_layers).to(device)

print(model)

WorldLSTM(
  (embedding): Embedding(11466, 128)
  (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=11466, bias=True)
)


In [6]:
# Optimization Setup & The Training Loop

lr = 0.001
num_epochs = 10

# Set up for the optimization criteria
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Change the model to training mode
model.train()

print('--- Training Loop Initialized ---')

for epoch in range(num_epochs):
    total_loss = 0.0

    for x, y in dataloader:
        # Send mini-batches to the active device
        x = x.to(device)
        y = y.to(device)

        # Clean the gradients from the last step
        optimizer.zero_grad()

        # Pass the data by the networl (Forward Pass)
        logits, _ = model(x)

        # Calculate the error by flatten the matrices from R2 to R and from R3 to R2
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))

        # Calculate the gradients (Backward Pass)
        loss.backward()

        # Cut the gradients (Gradient Clippling)
        # Max value of 1 for the gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # We update the weights of the model based on the gradients
        optimizer.step()

        # Stack the cost of the batch
        total_loss += loss.item()

    # Calculate teh average loss over the batches of the epoch
    avg_loss = total_loss / len(dataloader)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}')

--- Training Loop Initialized ---
Epoch 1/10, Loss: 5.0551
Epoch 2/10, Loss: 3.6856
Epoch 3/10, Loss: 2.6778
Epoch 4/10, Loss: 2.0144
Epoch 5/10, Loss: 1.5501
Epoch 6/10, Loss: 1.2272
Epoch 7/10, Loss: 1.0100
Epoch 8/10, Loss: 0.8674
Epoch 9/10, Loss: 0.7744
Epoch 10/10, Loss: 0.7115


In [9]:
# Autoregressive Inference (Text Generation Engine)

def generate_text(model, prompt, max_new_tokens=100, temperature=0.8):
    # We change the model to evaluation model
    model.eval()

    # We process teh initial phrase using the same tokenizer of RE
    prompt_tokens = re.findall(r'\w+|[^\w\s]', prompt.lower())

    # We convert the words from the prompt to IDs
    input_ids = [stoi[tok] for tok in prompt_tokens if tok in stoi]

    if len(input_ids) == 0:
        raise ValueError('None of the words from the initial phrase exists in the vocab')

    # convert the tensor of the batch adding an extra dimension
    x = torch.tensor([input_ids], dtype=torch.long).to(device)
    generated_tokens = prompt_tokens[:]
    hidden = None

    # We deactivaate the gradient calculation since we are just doing inference
    with torch.no_grad():
        # we input the full initial prompt
        logits, hidden = model(x, hidden)
        last_token = x[:, -1:] # We only extract the last token from the prompt

        for _ in range(max_new_tokens):
            # Predict based only in the last token and saved memory
            logits, hidden = model(last_token, hidden)

            # Apply temperature: controlling the creativity or randomness
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)

            # Multinomial sampling
            next_id = torch.multinomial(probs, num_samples=1)
            next_token = itos[next_id.item()]

            # Joint the new generated token to the history
            generated_tokens.append(next_token)

            # Update the last token for the next prediction cycle
            last_token = next_id

    # Static logic of the output text
    # Avoid leavinf empty spaces before the punctuation signs
    formatted_text = ""
    for tok in generated_tokens:
        if tok in '.,;:!?)]}':
            formatted_text += tok
        elif len(formatted_text) > 0 and formatted_text[-1] in "([{":
            formatted_text += tok
        else:
            if len(formatted_text) > 0:
              formatted_text += ' '
            formatted_text += tok

    return formatted_text

## Running a test with the prompt requested in the assignment
assignment_prompt = 'To be or not to be, this is the problem.'

# Generate the extension from shakespeare text
print('--- Text generation Target Prompt ---')
try:
    generated_text = generate_text(model, assignment_prompt, max_new_tokens=100, temperature=0.8)
    print('\n' + '=' * 80)
    print('Generated Text:')
    print('=' * 80)
    print(generated_text)
except NameError:
    print('Text cannot be generated until the model runs and the training from last step is completed')


--- Text generation Target Prompt ---

Generated Text:
to be or not to be, this is the problem. isabella: how doth her father? claudio: no, in such a case as any in blood as we are well resolved nor heard. angelo: will you speak o ' the people, and so i say, it had no heart to - morrow shall i do. stanley: who? brakenbury: in god ' s name what are you, and how shall we do our brother and his father, but wouldst gabble like a world by, makes my talk and our york edward duke; and, madam,
